# Day 09. Exercise 00
# Regularization

## 0. Imports

In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import pickle
import joblib

import sklearn as skl
import matplotlib

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {skl.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

NumPy: 1.19.5
Pandas: 1.1.5
Scikit-learn: 0.23.1
Matplotlib: 3.3.4


## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [25]:
df = pd.read_csv("../data/dayofweek.csv")

In [26]:
df.head()

,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,uid_user_16,uid_user_17,...,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,numTrials,hour,dayofweek
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.788667,-2.562352,4
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.756764,-2.562352,4
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.724861,-2.562352,4
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.692958,-2.562352,4
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.661055,-2.562352,4


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1686 entries, 0 to 1685
Data columns (total 44 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   uid_user_0        1686 non-null   float64
 1   uid_user_1        1686 non-null   float64
 2   uid_user_10       1686 non-null   float64
 3   uid_user_11       1686 non-null   float64
 4   uid_user_12       1686 non-null   float64
 5   uid_user_13       1686 non-null   float64
 6   uid_user_14       1686 non-null   float64
 7   uid_user_15       1686 non-null   float64
 8   uid_user_16       1686 non-null   float64
 9   uid_user_17       1686 non-null   float64
 10  uid_user_18       1686 non-null   float64
 11  uid_user_19       1686 non-null   float64
 12  uid_user_2        1686 non-null   float64
 13  uid_user_20       1686 non-null   float64
 14  uid_user_21       1686 non-null   float64
 15  uid_user_22       1686 non-null   float64
 16  uid_user_23       1686 non-null   float64


In [28]:
# Разделение на признаки и целевую переменную
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

# Разбиение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [29]:
%%time 

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    model = LogisticRegression(random_state=21, fit_intercept=False)
    model.fit(X_tr, y_tr)
    
    train_pred = model.predict(X_tr)
    valid_pred = model.predict(X_val)
    
    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)
    
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Average accuracy on crossval is {np.mean(valid_scores):.5f}')
print(f'Std is {np.std(valid_scores):.5f}')

train -  0.63974   |   valid -  0.65926
train -  0.63561   |   valid -  0.62222
train -  0.64386   |   valid -  0.60000
train -  0.64056   |   valid -  0.64444
train -  0.65375   |   valid -  0.60741
train -  0.62819   |   valid -  0.60000
train -  0.66035   |   valid -  0.60000
train -  0.63726   |   valid -  0.54074
train -  0.63756   |   valid -  0.66418
train -  0.64745   |   valid -  0.61194
Average accuracy on crossval is 0.61502
Std is 0.03399
CPU times: user 645 ms, sys: 5 ms, total: 650 ms
Wall time: 648 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [30]:

def cross_validate_model(penalty, solver):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
    valid_scores = []
    
    for train_index, valid_index in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]
        
        model = LogisticRegression(
            penalty=penalty, 
            solver=solver, 
            random_state=21, 
            fit_intercept=False,
            max_iter=1000
        )
        model.fit(X_tr, y_tr)
        
        valid_pred = model.predict(X_val)
        valid_acc = accuracy_score(y_val, valid_pred)
        valid_scores.append(valid_acc)
    
    mean_acc = np.mean(valid_scores)
    std_acc = np.std(valid_scores)
    print(f'{penalty} + {solver}: mean={mean_acc:.5f}, std={std_acc:.5f}')
    return mean_acc


In [31]:
print('=== penalty=none ===')
cross_validate_model('none', 'lbfgs')
cross_validate_model('none', 'sag')
cross_validate_model('none', 'saga')
cross_validate_model('none', 'newton-cg')

print('\n=== penalty=l1 ===')
cross_validate_model('l1', 'liblinear')
cross_validate_model('l1', 'saga')

print('\n=== penalty=l2 ===')
cross_validate_model('l2', 'liblinear')
cross_validate_model('l2', 'lbfgs')
cross_validate_model('l2', 'sag')
cross_validate_model('l2', 'saga')
cross_validate_model('l2', 'newton-cg');

=== penalty=none ===
none + lbfgs: mean=0.63432, std=0.03340


/home/user/goinfre/miniconda3/envs/DSB10/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/home/user/goinfre/miniconda3/envs/DSB10/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/home/user/goinfre/miniconda3/envs/DSB10/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/home/user/goinfre/miniconda3/envs/DSB10/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/home/user/goinfre/miniconda

none + sag: mean=0.63432, std=0.03340


/home/user/goinfre/miniconda3/envs/DSB10/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/home/user/goinfre/miniconda3/envs/DSB10/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/home/user/goinfre/miniconda3/envs/DSB10/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/home/user/goinfre/miniconda3/envs/DSB10/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/home/user/goinfre/miniconda

none + saga: mean=0.63432, std=0.03340
none + newton-cg: mean=0.63432, std=0.03340

=== penalty=l1 ===
l1 + liblinear: mean=0.60240, std=0.03627
l1 + saga: mean=0.60758, std=0.03367

=== penalty=l2 ===
l2 + liblinear: mean=0.59945, std=0.02701
l2 + lbfgs: mean=0.61502, std=0.03399
l2 + sag: mean=0.61502, std=0.03399
l2 + saga: mean=0.61502, std=0.03399
l2 + newton-cg: mean=0.61502, std=0.03399


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [32]:
%%time

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    model = SVC(probability=True, kernel='linear', random_state=21)
    model.fit(X_tr, y_tr)
    
    train_pred = model.predict(X_tr)
    valid_pred = model.predict(X_val)
    
    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)
    
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Average accuracy on crossval is {np.mean(valid_scores):.5f}')
print(f'Std is {np.std(valid_scores):.5f}')

train -  0.70651   |   valid -  0.68148
train -  0.68920   |   valid -  0.64444
train -  0.69744   |   valid -  0.66667
train -  0.68920   |   valid -  0.65926
train -  0.69497   |   valid -  0.63704
train -  0.68673   |   valid -  0.68148
train -  0.69827   |   valid -  0.61481
train -  0.70486   |   valid -  0.57778
train -  0.68863   |   valid -  0.72388
train -  0.71005   |   valid -  0.64179
Average accuracy on crossval is 0.65286
Std is 0.03800
CPU times: user 3.52 s, sys: 3.99 ms, total: 3.52 s
Wall time: 3.51 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [33]:
def cross_validate_svm(C):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
    valid_scores = []
    
    for train_index, valid_index in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]
        
        model = SVC(C=C, probability=True, kernel='linear', random_state=21)
        model.fit(X_tr, y_tr)
        
        valid_pred = model.predict(X_val)
        valid_acc = accuracy_score(y_val, valid_pred)
        valid_scores.append(valid_acc)
    
    mean_acc = np.mean(valid_scores)
    std_acc = np.std(valid_scores)
    print(f'C={C:8.4f}: mean={mean_acc:.5f}, std={std_acc:.5f}')
    return mean_acc

In [34]:
# Тестирование разных значений C
print('=== SVM C values ===')
cross_validate_svm(0.001)
cross_validate_svm(0.01)
cross_validate_svm(0.1)
cross_validate_svm(1.0)
cross_validate_svm(10.0)
cross_validate_svm(100.0)

=== SVM C values ===
C=  0.0010: mean=0.23443, std=0.00397
C=  0.0100: mean=0.38125, std=0.02293
C=  0.1000: mean=0.55787, std=0.02522
C=  1.0000: mean=0.65286, std=0.03800
C= 10.0000: mean=0.71814, std=0.04026
C=100.0000: mean=0.75001, std=0.02849


0.7500110558319515

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [35]:
%%time

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    model = DecisionTreeClassifier(max_depth=10, random_state=21)
    model.fit(X_tr, y_tr)
    
    train_pred = model.predict(X_tr)
    valid_pred = model.predict(X_val)
    
    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)
    
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Average accuracy on crossval is {np.mean(valid_scores):.5f}')
print(f'Std is {np.std(valid_scores):.5f}')

train -  0.80874   |   valid -  0.77037
train -  0.79802   |   valid -  0.70370
train -  0.81286   |   valid -  0.73333
train -  0.80049   |   valid -  0.74815
train -  0.80956   |   valid -  0.70370
train -  0.78978   |   valid -  0.75556
train -  0.80709   |   valid -  0.62963
train -  0.82688   |   valid -  0.71111
train -  0.78995   |   valid -  0.79851
train -  0.80313   |   valid -  0.70896
Average accuracy on crossval is 0.72630
Std is 0.04409
CPU times: user 134 ms, sys: 8 µs, total: 134 ms
Wall time: 133 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [36]:
def cross_validate_tree(max_depth, min_samples_split=2, min_samples_leaf=1):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
    valid_scores = []
    
    for train_index, valid_index in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]
        
        model = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=21
        )
        model.fit(X_tr, y_tr)
        
        valid_pred = model.predict(X_val)
        valid_acc = accuracy_score(y_val, valid_pred)
        valid_scores.append(valid_acc)
    
    mean_acc = np.mean(valid_scores)
    std_acc = np.std(valid_scores)

    # Обработка None для вывода
    if max_depth is None:
        print(f'max_depth=None: mean={mean_acc:.5f}, std={std_acc:.5f}')
    else:
        print(f'max_depth={max_depth:2d}: mean={mean_acc:.5f}, std={std_acc:.5f}')
        
    return mean_acc

In [37]:
# Тестирование разных значений max_depth
print('=== max_depth values ===')
cross_validate_tree(1)
cross_validate_tree(3)
cross_validate_tree(5)
cross_validate_tree(7)
cross_validate_tree(10)
cross_validate_tree(15)
cross_validate_tree(20)
cross_validate_tree(None);

=== max_depth values ===
max_depth= 1: mean=0.35530, std=0.02063
max_depth= 3: mean=0.47031, std=0.01864
max_depth= 5: mean=0.54522, std=0.04134
max_depth= 7: mean=0.65210, std=0.04108
max_depth=10: mean=0.72630, std=0.04409
max_depth=15: mean=0.85831, std=0.02685
max_depth=20: mean=0.88649, std=0.01911
max_depth=None: mean=0.88427, std=0.02179


In [38]:

def cross_validate_tree(max_depth, min_samples_split=2, min_samples_leaf=1):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
    valid_scores = []
    
    for train_index, valid_index in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]
        
        model = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=21
        )
        model.fit(X_tr, y_tr)
        
        valid_pred = model.predict(X_val)
        valid_acc = accuracy_score(y_val, valid_pred)
        valid_scores.append(valid_acc)
    
    return np.mean(valid_scores)

# Поиск лучшей комбинации
print('=== Bonus: combinations ===')
best_score = 0
best_params = {}

for depth in [10, 15, 20, None]:
    for split in [2, 5, 10]:
        for leaf in [1, 2, 5]:
            score = cross_validate_tree(depth, split, leaf)
            if score > best_score:
                best_score = score
                best_params = {'max_depth': depth, 'min_samples_split': split, 'min_samples_leaf': leaf}
            print(f'depth={str(depth):4s}, split={split:2d}, leaf={leaf}: {score:.5f}')

print(f'\nBest params: {best_params}')
print(f'Best score: {best_score:.5f}')

=== Bonus: combinations ===
depth=10  , split= 2, leaf=1: 0.72630
depth=10  , split= 2, leaf=2: 0.70627
depth=10  , split= 2, leaf=5: 0.68104
depth=10  , split= 5, leaf=1: 0.71889
depth=10  , split= 5, leaf=2: 0.70849
depth=10  , split= 5, leaf=5: 0.68104
depth=10  , split=10, leaf=1: 0.70256
depth=10  , split=10, leaf=2: 0.69589
depth=10  , split=10, leaf=5: 0.68104
depth=15  , split= 2, leaf=1: 0.85831
depth=15  , split= 2, leaf=2: 0.82195
depth=15  , split= 2, leaf=5: 0.77373
depth=15  , split= 5, leaf=1: 0.85161
depth=15  , split= 5, leaf=2: 0.82787
depth=15  , split= 5, leaf=5: 0.77373
depth=15  , split=10, leaf=1: 0.82640
depth=15  , split=10, leaf=2: 0.81527
depth=15  , split=10, leaf=5: 0.77373
depth=20  , split= 2, leaf=1: 0.88649
depth=20  , split= 2, leaf=2: 0.84716
depth=20  , split= 2, leaf=5: 0.79153
depth=20  , split= 5, leaf=1: 0.87906
depth=20  , split= 5, leaf=2: 0.85087
depth=20  , split= 5, leaf=5: 0.79153
depth=20  , split=10, leaf=1: 0.84494
depth=20  , split=10, 

## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [39]:
%%time
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=14,
        random_state=21
    )
    model.fit(X_tr, y_tr)
    
    train_pred = model.predict(X_tr)
    valid_pred = model.predict(X_val)
    
    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)
    
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Average accuracy on crossval is {np.mean(valid_scores):.5f}')
print(f'Std is {np.std(valid_scores):.5f}')

train -  0.97362   |   valid -  0.86667
train -  0.96867   |   valid -  0.83704
train -  0.96455   |   valid -  0.89630
train -  0.97197   |   valid -  0.92593
train -  0.97115   |   valid -  0.88889
train -  0.95548   |   valid -  0.89630
train -  0.96373   |   valid -  0.88148
train -  0.97362   |   valid -  0.88148
train -  0.97117   |   valid -  0.92537
train -  0.96458   |   valid -  0.85821
Average accuracy on crossval is 0.88577
Std is 0.02636
CPU times: user 1.01 s, sys: 7.01 ms, total: 1.02 s
Wall time: 1.02 s


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [40]:

def cross_validate_rf(max_depth, n_estimators, min_samples_split=2, min_samples_leaf=1, max_features='sqrt'):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
    valid_scores = []
    
    for train_index, valid_index in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]
        
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=21
        )
        model.fit(X_tr, y_tr)
        
        valid_pred = model.predict(X_val)
        valid_acc = accuracy_score(y_val, valid_pred)
        valid_scores.append(valid_acc)
    
    return np.mean(valid_scores)

# Перебор max_depth и n_estimators
print('=== RF: max_depth vs n_estimators ===')
best_score = 0
best_params = {}

for depth in [10, 14, 18, 20, None]:
    for estimators in [20, 50, 100]:
        score = cross_validate_rf(depth, estimators)
        if score > best_score:
            best_score = score
            best_params = {'max_depth': depth, 'n_estimators': estimators}
        print(f'depth={str(depth):4s}, estimators={estimators:3d}: {score:.5f}')

print(f'\nBest params: {best_params}')
print(f'Best score: {best_score:.5f}')

=== RF: max_depth vs n_estimators ===
depth=10  , estimators= 20: 0.79526
depth=10  , estimators= 50: 0.80787
depth=10  , estimators=100: 0.80045
depth=14  , estimators= 20: 0.87985
depth=14  , estimators= 50: 0.88577
depth=14  , estimators=100: 0.88872
depth=18  , estimators= 20: 0.89171
depth=18  , estimators= 50: 0.90431
depth=18  , estimators=100: 0.90654
depth=20  , estimators= 20: 0.90209
depth=20  , estimators= 50: 0.91247
depth=20  , estimators=100: 0.91543
depth=None, estimators= 20: 0.90653
depth=None, estimators= 50: 0.91542
depth=None, estimators=100: 0.91618

Best params: {'max_depth': None, 'n_estimators': 100}
Best score: 0.91618


In [41]:
# Расширенный поиск лучшей комбинации
print('=== RF Bonus: full combinations ===')
best_score = 0
best_params = {}

for depth in [18, 20, None]:
    for estimators in [50, 100]:
        for split in [2, 5]:
            for leaf in [1, 2]:
                for features in ['sqrt', 'log2']:
                    score = cross_validate_rf(depth, estimators, split, leaf, features)
                    if score > best_score:
                        best_score = score
                        best_params = {
                            'max_depth': depth,
                            'n_estimators': estimators,
                            'min_samples_split': split,
                            'min_samples_leaf': leaf,
                            'max_features': features
                        }
                    print(f'depth={str(depth):4s}, est={estimators:3d}, split={split}, leaf={leaf}, feat={features}: {score:.5f}')

print(f'\nBest params: {best_params}')
print(f'Best score: {best_score:.5f}')

=== RF Bonus: full combinations ===
depth=18  , est= 50, split=2, leaf=1, feat=sqrt: 0.90431
depth=18  , est= 50, split=2, leaf=1, feat=log2: 0.90133
depth=18  , est= 50, split=2, leaf=2, feat=sqrt: 0.85683
depth=18  , est= 50, split=2, leaf=2, feat=log2: 0.85013
depth=18  , est= 50, split=5, leaf=1, feat=sqrt: 0.89987
depth=18  , est= 50, split=5, leaf=1, feat=log2: 0.89540
depth=18  , est= 50, split=5, leaf=2, feat=sqrt: 0.85013
depth=18  , est= 50, split=5, leaf=2, feat=log2: 0.85090
depth=18  , est=100, split=2, leaf=1, feat=sqrt: 0.90654
depth=18  , est=100, split=2, leaf=1, feat=log2: 0.90281
depth=18  , est=100, split=2, leaf=2, feat=sqrt: 0.85606
depth=18  , est=100, split=2, leaf=2, feat=log2: 0.85608
depth=18  , est=100, split=5, leaf=1, feat=sqrt: 0.89910
depth=18  , est=100, split=5, leaf=1, feat=log2: 0.89391
depth=18  , est=100, split=5, leaf=2, feat=sqrt: 0.85310
depth=18  , est=100, split=5, leaf=2, feat=log2: 0.85162
depth=20  , est= 50, split=2, leaf=1, feat=sqrt: 0.9

In [42]:
# Сравнение разных значений max_features
for features in ['sqrt', 'log2', 10, 20, None]:
    score = cross_validate_rf(max_depth=None, n_estimators=100, max_features=features)
    print(f'max_features={str(features):6s}: {score:.5f}')

max_features=sqrt  : 0.91618
max_features=log2  : 0.91469
max_features=10    : 0.91470
max_features=20    : 0.91468
max_features=None  : 0.90505


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [43]:

# Обучение лучшей модели на всём X_train
best_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    max_features='sqrt',
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=21
)
best_model.fit(X_train, y_train)

# Предсказания на тесте
y_pred = best_model.predict(X_test)

# Финальная точность
test_accuracy = accuracy_score(y_test, y_pred)
print(f'Final test accuracy: {test_accuracy:.5f}')

# Анализ ошибок по дням недели
df_results = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred})
df_results['is_error'] = df_results['y_true'] != df_results['y_pred']

# Процент ошибок по каждому классу
error_by_class = df_results.groupby('y_true')['is_error'].mean() * 100
print('\nError rate by weekday:')
for day, error in error_by_class.items():
    print(f'Day {day}: {error:.2f}%')

# День с наибольшим процентом ошибок
worst_day = error_by_class.idxmax()
worst_error = error_by_class.max()
print(f'\nWorst day: {worst_day} with {worst_error:.2f}% errors')

# Сохранение модели
joblib.dump(best_model, '../data/best_model.pkl')
print('Model saved to ../data/best_model.pkl')


Final test accuracy: 0.93787

Error rate by weekday:
Day 0: 25.93%
Day 1: 5.45%
Day 2: 10.00%
Day 3: 2.50%
Day 4: 9.52%
Day 5: 5.56%
Day 6: 1.41%

Worst day: 0 with 25.93% errors
Model saved to ../data/best_model.pkl


In [44]:
# проверка равномерности распределения данных по дням
# Количество примеров каждого класса в тесте
print('Samples per class in test:')
print(y_test.value_counts().sort_index())

# Сколько ошибок по абсолютному числу
df_results = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred})
df_results['is_error'] = df_results['y_true'] != df_results['y_pred']
errors_count = df_results[df_results['is_error']].groupby('y_true').size()
print('\nAbsolute errors per class:')
print(errors_count)

Samples per class in test:
0    27
1    55
2    30
3    80
4    21
5    54
6    71
Name: dayofweek, dtype: int64

Absolute errors per class:
y_true
0    7
1    3
2    3
3    2
4    2
5    3
6    1
dtype: int64


In [45]:
# Модель с автоматическим учётом дисбаланса
balanced_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    max_features='sqrt',
    class_weight='balanced',  # Автоматически взвешивает классы
    random_state=21
)
balanced_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_features='sqrt',
                       random_state=21)

In [46]:
from sklearn.metrics import classification_report

# Модель без весов
model1 = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=21)
model1.fit(X_train, y_train)
pred1 = model1.predict(X_test)
print('Without class_weight:')
print(classification_report(y_test, pred1))

# Модель с весами
model2 = RandomForestClassifier(n_estimators=100, max_depth=None, class_weight='balanced', random_state=21)
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)
print('With class_weight=balanced:')
print(classification_report(y_test, pred2))

Without class_weight:
              precision    recall  f1-score   support

           0       0.91      0.74      0.82        27
           1       0.96      0.95      0.95        55
           2       0.96      0.90      0.93        30
           3       0.94      0.97      0.96        80
           4       1.00      0.90      0.95        21
           5       0.91      0.94      0.93        54
           6       0.92      0.99      0.95        71

    accuracy                           0.94       338
   macro avg       0.94      0.91      0.93       338
weighted avg       0.94      0.94      0.94       338

With class_weight=balanced:
              precision    recall  f1-score   support

           0       0.95      0.74      0.83        27
           1       0.96      0.96      0.96        55
           2       1.00      0.93      0.97        30
           3       0.94      0.96      0.95        80
           4       1.00      0.86      0.92        21
           5       0.89     